# Day 6 — SQL Practice Questions: GROUP BY, ROLLUP, GROUPING SETS

**3 questions** · Easy → Medium  
Tables are created inline — no DBeaver setup required.

In [ ]:
%load_ext sql

USERNAME = 'postgres'
PASSWORD = 'hariom'
HOST     = 'localhost'
PORT     = '5432'
DBNAME   = 'postgres'

%sql postgresql://{USERNAME}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}

In [ ]:
%%sql
DROP TABLE IF EXISTS d6_sales CASCADE;

CREATE TABLE d6_sales (
    sale_id   SERIAL PRIMARY KEY,
    region    VARCHAR(20),
    category  VARCHAR(20),
    product   VARCHAR(30),
    sale_date DATE,
    revenue   NUMERIC(10, 2)
);

INSERT INTO d6_sales (region, category, product, sale_date, revenue) VALUES
  ('North', 'Electronics', 'Laptop',   '2024-01-05', 1200.00),
  ('North', 'Electronics', 'Phone',    '2024-01-10',  800.00),
  ('North', 'Furniture',   'Chair',    '2024-01-12',  300.00),
  ('North', 'Furniture',   'Desk',     '2024-01-15',  600.00),
  ('South', 'Electronics', 'Tablet',   '2024-01-08',  500.00),
  ('South', 'Electronics', 'Laptop',   '2024-01-20', 1100.00),
  ('South', 'Furniture',   'Sofa',     '2024-01-18',  900.00),
  ('South', 'Furniture',   'Wardrobe', '2024-01-22', 1500.00),
  ('East',  'Electronics', 'Phone',    '2024-01-09',  750.00),
  ('East',  'Furniture',   'Chair',    '2024-01-14',  280.00);

SELECT 'Tables created' AS status;

---
## Q1 (Easy) — Revenue by Region and Category

Write a query that returns the **total revenue** and **number of sales** grouped by `region` and `category`, ordered by `region` then `category`.

Expected: 5 rows (North-Electronics, North-Furniture, South-Electronics, South-Furniture, East-Electronics, East-Furniture... wait, how many unique combinations are there?)

**Warm-ups:**
- How many distinct (region, category) pairs are in the data?
- What happens if you GROUP BY only `region`? How many rows?
- What aggregate function counts non-NULL values?

In [ ]:
%%sql
-- YOUR QUERY HERE
-- Expected columns: region, category, num_sales, total_revenue
-- Expected rows: one per (region, category) combination

### Solution

In [ ]:
%%sql
-- SOLUTION
SELECT   region,
         category,
         COUNT(*)     AS num_sales,
         SUM(revenue) AS total_revenue
FROM     d6_sales
GROUP BY region, category
ORDER BY region, category;
-- 6 rows: East×2, North×2, South×2

---
## Q2 (Medium) — ROLLUP with Labeled Subtotals

Write a ROLLUP query over `(region, category)` that includes:
- Detail rows per (region, category)
- Subtotal per region (NULL category = subtotal row)
- Grand total row

Replace NULL values with `'ALL REGIONS'` / `'ALL CATEGORIES'` so the output is readable.

**Expected:** 10 detail rows + 3 region subtotals + 1 grand total = 14 rows

**Warm-ups:**
- What does ROLLUP(region, category) produce that plain GROUP BY does not?
- How do you replace a rollup-NULL with a readable label?
- How many rows does ROLLUP add on top of plain GROUP BY?

In [ ]:
%%sql
-- YOUR QUERY HERE
-- Use ROLLUP(region, category)
-- Replace NULLs with 'ALL REGIONS' / 'ALL CATEGORIES'

### Solution

In [ ]:
%%sql
-- SOLUTION
SELECT
    COALESCE(region,   'ALL REGIONS')    AS region,
    COALESCE(category, 'ALL CATEGORIES') AS category,
    COUNT(*)     AS num_sales,
    SUM(revenue) AS total_revenue
FROM   d6_sales
GROUP BY ROLLUP(region, category)
ORDER BY region NULLS LAST, category NULLS LAST;
-- 6 detail + 3 region subtotals + 1 grand total = 10 rows

---
## Q3 (Medium) — GROUPING SETS with Level Label

Write a query using `GROUPING SETS` that produces:
1. Revenue grouped by `(region, category)` — detail
2. Revenue grouped by `(region)` only — region subtotals
3. Revenue grouped by `(category)` only — category subtotals

Add a `grouping_level` column that labels each row as `'region+category'`, `'region only'`, or `'category only'`.

**Note:** No grand total row — only include the 3 levels above.

**Warm-ups:**
- Why can't ROLLUP produce both region-only AND category-only subtotals in one query?
- What does `GROUPING(region)` return for a region-only subtotal row?
- What does `GROUPING(category)` return for a region+category detail row?

In [ ]:
%%sql
-- YOUR QUERY HERE
-- Use GROUPING SETS with 3 levels (no grand total)
-- Add grouping_level column using CASE WHEN GROUPING()

### Solution

In [ ]:
%%sql
-- SOLUTION
SELECT
    COALESCE(region,   'ALL')  AS region,
    COALESCE(category, 'ALL')  AS category,
    SUM(revenue)               AS total_revenue,
    CASE
        WHEN GROUPING(region) = 0 AND GROUPING(category) = 0 THEN 'region+category'
        WHEN GROUPING(region) = 0                             THEN 'region only'
        ELSE                                                       'category only'
    END AS grouping_level
FROM   d6_sales
GROUP  BY GROUPING SETS (
    (region, category),
    (region),
    (category)
)
ORDER BY grouping_level, region NULLS LAST, category NULLS LAST;
-- 6 detail + 3 region-only + 2 category-only = 11 rows